In [3]:
pip install -q faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
import re
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
### ETAPA DE CARREAGMENTO DO DOCUMENTO

In [12]:
loader = TextLoader(
    file_path=r"C:\Users\Mateus\Desktop\Udemy\Complete Agentic AI Bootcamp\Secao 5 - Langchain hands On\Data_Ingestion\solucoes_ti_rag.txt",
    encoding="utf-8"
)

docs = loader.load()

In [31]:
### ETAPA DE LIMPEZA DO DOCUMENTO, CARACTERES COMO '=====', '----' E OUTROS, VÃO ATRABALHAR O CALCULO DE SIMILARIDADE DE COSSENO

In [32]:
import re


def limpar_texto(texto):
    texto = re.sub(r'^[=\-_*]{3,}\s*$', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'[=\-_*]{3,}', '', texto)
    texto = re.sub(r'\n\s*\n', '\n\n', texto)
    return texto.strip()

for doc in docs:
    doc.page_content = limpar_texto(doc.page_content)

In [33]:
### ETAPA DE SEPARAÇÃO EM BLOCOS DE TEXTOS (CHUNKS JÁ LIMPOS)

In [40]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1100,
    chunk_overlap = 200
)


chunks = splitter.split_documents(docs)

In [41]:
for i in range(0,12):
    print(f'Chunk -> {i}')
    print(chunks[i].page_content)
    print('/////////////////////////')

Chunk -> 0
MANUAL E BASE DE CONHECIMENTO DE SOLUÇÕES DE INFRAESTRUTURA DE TI

1. ARQUITETURA DE COMPUTACÃO EM NUVEM E HÍBRIDA

1.1 Nuvem Pública, Privada e Híbrida
A infraestrutura de TI moderna baseia-se na flexibilidade do modelo de computação em nuvem. 
- Nuvem Pública: Provedores como AWS, Microsoft Azure e Google Cloud Platform (GCP) oferecem recursos computacionais compartilhados sobre a internet. Oferece alta escalabilidade, modelo de pagamento conforme o uso (OpEx) e redução de investimento inicial em hardware.
- Nuvem Privada: Ambiente computacional dedicado exclusivamente a uma única organização. Pode ser hospedado on-premise no datacenter local ou por um terceiro. Garante maior controle sobre conformidade, segurança e isolamento de dados.
- Nuvem Híbrida: Integração entre infraestruturas on-premise, nuvem privada e nuvem pública. Permite que dados e aplicações sejam compartilhados entre elas, otimizando cargas de trabalho com base em requisitos de segurança, custo e desempen

In [46]:
### ETAPA DE VETORIZAÇÃO DOS DOCUMENTOS (EMBEDDINGS)

In [45]:
pip install -qU langchain-ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [83]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
db = FAISS.from_documents(chunks,embeddings)
db

In [84]:
query = """
Quais ferramentas são recomendadas para orquestração de contêineres e para automação de infraestrutura como código (IaC)?
"""

k = 3
retorno_busca = db.similarity_search(query,k=k)

for a in range(k):
    print(retorno_busca[a].page_content)

6.2 Práticas de DevOps e Infraestrutura como Código (IaC)
- Infraestrutura como Código (IaC): Provisionamento e gerenciamento de recursos de infraestrutura por meio de código declarativo em vez de processos manuais. Garante reprodutibilidade, controle de versão e consistência entre ambientes de dev, test e prod. Ferramentas: Terraform, OpenTofu, AWS CloudFormation.
- Gerenciamento de Configuração: Automação da instalação de softwares, patches e configurações em servidores em massa. Ferramentas: Ansible, Puppet, Chef.
MANUAL E BASE DE CONHECIMENTO DE SOLUÇÕES DE INFRAESTRUTURA DE TI

1. ARQUITETURA DE COMPUTACÃO EM NUVEM E HÍBRIDA

1.1 Nuvem Pública, Privada e Híbrida
A infraestrutura de TI moderna baseia-se na flexibilidade do modelo de computação em nuvem. 
- Nuvem Pública: Provedores como AWS, Microsoft Azure e Google Cloud Platform (GCP) oferecem recursos computacionais compartilhados sobre a internet. Oferece alta escalabilidade, modelo de pagamento conforme o uso (OpEx) e redução 

#  Entendendo o `.as_retriever()` no LangChain

No LangChain, o objeto **FAISS** é um **`VectorStore`** (armazenamento e indexação de vetores). 

Ao chamar o método `.as_retriever()`, transformamos esse banco de dados vetorial em uma interface padronizada chamada **`Retriever`** (Recuperador).

---

###  Por que converter um `VectorStore` para `Retriever`?

Existem **3 motivos principais** para essa abstração em arquiteturas RAG reais:

#### 1. Integração Nativa com Chains (LCEL)
O `VectorStore` puro (`db`) exige chamadas manuais como `db.similarity_search()`. Já o `Retriever` foi projetado para se conectar diretamente à **LangChain Expression Language (LCEL)**, integrando-se nativamente com `Prompts` e `LLMs` em uma única cadeia:

```python
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)
```

#### 2. Padronização do Protocolo Runnable (`.invoke()`)
Nas versões modernas do LangChain, todos os componentes principais seguem a interface **`Runnable`**, utilizando o método padrão `.invoke()`. 

Ao executar `retriever.invoke(query)`, recebemos a lista de objetos `Document` de forma consistente e compatível com todo o ecossistema.

#### 3. Flexibilidade na Configuração da Busca
Podemos definir estratégias de busca avançadas diretamente na inicialização do retriever via `search_kwargs`:

```python
#  Busca padrão por Similaridade (Top-K)
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

#  Busca com MMR (Maximal Marginal Relevance) para evitar informações redundantes
retriever_mmr = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)
```

In [73]:
retriever = db.as_retriever()

print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [75]:
result = retriever.invoke(query)

print(result)

[Document(id='c7184009-f8e2-4698-8626-61d2b29261c1', metadata={'source': 'C:\\Users\\Mateus\\Desktop\\Udemy\\Complete Agentic AI Bootcamp\\Secao 5 - Langchain hands On\\Data_Ingestion\\solucoes_ti_rag.txt'}, page_content='6.2 Práticas de DevOps e Infraestrutura como Código (IaC)\n- Infraestrutura como Código (IaC): Provisionamento e gerenciamento de recursos de infraestrutura por meio de código declarativo em vez de processos manuais. Garante reprodutibilidade, controle de versão e consistência entre ambientes de dev, test e prod. Ferramentas: Terraform, OpenTofu, AWS CloudFormation.\n- Gerenciamento de Configuração: Automação da instalação de softwares, patches e configurações em servidores em massa. Ferramentas: Ansible, Puppet, Chef.'), Document(id='c9423d29-a39c-48ad-941d-ba7cd0ff114b', metadata={'source': 'C:\\Users\\Mateus\\Desktop\\Udemy\\Complete Agentic AI Bootcamp\\Secao 5 - Langchain hands On\\Data_Ingestion\\solucoes_ti_rag.txt'}, page_content='MANUAL E BASE DE CONHECIMENTO

In [76]:
### busca similaridade por score
### there are some FAISS specific methods, one of'em is similarity_seach_with_score, which allows you to return not only the documents 
### but also the distance score of the query to them. 
### The returned distance score is L2 distance. therefore a lower distance is better

# 📊 Busca por Similaridade com Score (`similarity_search_with_score`)

O FAISS possui métodos específicos como o `similarity_search_with_score`, que retorna não apenas os documentos relevantes, mas também o **score de distância** entre a pergunta e cada trecho encontrado.

---

### 📏 O que significa a pontuação (Distância L2)?

O FAISS utiliza por padrão a **Distância Euclidiana ($L_2$)** para medir a distância vetorial entre a consulta e o texto armazenado.

> **Regra de Ouro:** Quanto **MENOR** o score, **MAIOR** é a relevância semântica do documento.

* **Score = $0.0$:** O texto do documento é idêntico à pergunta (distância zero).
* **Score Baixo:** Alta relevância semântica. O trecho responde diretamente à dúvida.
* **Score Alto:** Pouca ou nenhuma relevância. O documento está distante do conceito buscado.

---

### 💡 Por que utilizar o Score no RAG?

1. **Definição de Threshold (Limiar de Corte):** Permite filtrar resultados e descartar documentos com distância elevada antes de enviá-los ao LLM.
2. **Prevenção de Alucinações:** Se todos os trechos retornarem scores muito altos, o sistema pode responder diretamente ao usuário que não encontrou a informação na base de conhecimento.

In [88]:
pergunta = "Como é a virtualização e contêiners da empresa?"

similarity_score = db.similarity_search_with_score(pergunta,k=k)

for doc, score in similarity_score:
    print(f"Score (Distância L2): {score:.4f}")
    print(f"Conteúdo: {doc.page_content}...\n")
    print('************************************************************************')

Score (Distância L2): 322.6978
Conteúdo: 2. VIRTUALIZACÃO E CONTÊINERES

2.1 Hipervisores e Virtualização de Servidores
A virtualização permite a criação de múltiplas instâncias virtuais em um único hardware físico através de um hipervisor.
- Type-1 (Bare-Metal): O hipervisor é instalado diretamente no hardware físico. Oferece alto desempenho, menor latência e maior segurança. Exemplos: VMware ESXi, Microsoft Hyper-V, Proxmox VE, KVM.
- Type-2 (Hospedado): O hipervisor roda sobre um sistema operacional hospedeiro. Usado principalmente para testes e ambientes de desenvolvimento. Exemplos: Oracle VirtualBox, VMware Workstation....

************************************************************************
Score (Distância L2): 339.9733
Conteúdo: 2.2 Contêineres e Orquestração
Diferente das máquinas virtuais que virtualizam o hardware e exigem um SO completo, os contêineres virtualizam o sistema operacional, compartilhando o mesmo kernel e isolando apenas as bibliotecas e dependências da ap

### SALVANDO E CARREGANDO O NOSSO BANCO DE DADOS VETORIAIS LOCALMENTE, PARA USAR POSTERIORMENTE EM OUTROS NOTEBOOKS

In [89]:
db.save_local("faiss_index")

In [94]:
new_db = FAISS.load_local(r"C:\Users\Mateus\Desktop\Udemy\Complete Agentic AI Bootcamp\Secao 5 - Langchain hands On\faiss_index",
                          embeddings=embeddings,
                         allow_dangerous_deserialization=True
)


seach = new_db.similarity_search_with_score(
    query="Relate sobre ferramentas de observabilidade e monitoramento"
)


for doc, score in seach:
    print(f"Score (Distância L2): {score:.4f}")
    print(f"Conteúdo: {doc.page_content}...\n")
    print('************************************************************************')

Score (Distância L2): 349.1399
Conteúdo: 6. MONITORAMENTO E OPERAÇÕES DE TI (ITOps)

6.1 Ferramentas de Observabilidade e Monitoramento
- Monitoramento de Infraestrutura: Coleta de métricas de CPU, memória, uso de disco e tráfego de rede usando agentes SNMP ou agens proprietários. Ferramentas comuns: Zabbix, Prometheus, Grafana, Nagios.
- APM (Application Performance Monitoring): Monitoramento detalhado do desempenho do código da aplicação, consultas SQL e transações do usuário. Ferramentas: Datadog, Dynatrace, New Relic.
- SIEM (Security Information and Event Management): Centralização e análise de logs de segurança de toda a infraestrutura para correlação de eventos e resposta a incidentes (ex.: Splunk, Elastic Security, Microsoft Sentinel)....

************************************************************************
Score (Distância L2): 391.6209
Conteúdo: 4. ARMAZENAMENTO DE DADOS (STORAGE)

4.1 Arquiteturas de Storage
- SAN (Storage Area Network): Rede dedicada de alta velocidade 